In [1]:
%load_ext autoreload
%autoreload 2

from quantem.core import config

config.set_device(0)
print(config.get("device"))

import pickle

import numpy as np
import torch
from matplotlib import pyplot as plt

from stem_utils import generate_datacube

ACC_VOLTAGE     = 60e3   # eV
PROBE_SEMIANGLE = 20     # mrad
PROBE_DEFOCUS   = 30     # Å
SCAN_STEP_SIZE  = 0.1   # Å
DP_TRIM         = 30     # pixels trimmed off each side of the detector

with open('DIP_phases_wide_stich.pkl', 'rb') as f:
    data = pickle.load(f)

x = data['x']
y = data['y']

cuda:0


In [3]:
def simulate_clean_datacube(phase_image, x=x, y=y):
    """Noise-free 4D-STEM simulation of a single phase image on the shared scan grid."""
    return generate_datacube(
        phase_image, x, y,
        acc_voltage=ACC_VOLTAGE,
        probe_semiangle=PROBE_SEMIANGLE,
        probe_defocus=PROBE_DEFOCUS,
        scan_step_size=SCAN_STEP_SIZE,
    )


def checkerboard_mask(shape):
    ii, jj = np.indices(shape)
    return ((ii + jj) % 2 == 0).astype(np.uint8)


def merge_scan(meas_a, meas_b, scan_mask, dose=None, dp_trim=DP_TRIM, normalize=True):
    """Interlace two noise-free measurements on a checkerboard scan pattern.

    `dose` (electrons / Å^2) applies Poisson shot noise to each measurement
    before interlacing; `None` keeps the simulation noise-free.
    """
    m_a = meas_a if dose is None else meas_a.poisson_noise(dose)
    m_b = meas_b if dose is None else meas_b.poisson_noise(dose)

    dcube_a = np.array(m_a.array)[:, :, dp_trim:-dp_trim, dp_trim:-dp_trim]
    dcube_b = np.array(m_b.array)[:, :, dp_trim:-dp_trim, dp_trim:-dp_trim]

    merged_scan = np.where(scan_mask[..., np.newaxis, np.newaxis] == 1, dcube_b, dcube_a)

    if normalize:
        merged_scan = merged_scan / merged_scan.sum(axis=(2, 3), keepdims=True)

    return merged_scan

In [4]:
baseline_meas = simulate_clean_datacube(data['stitched'][0])
excited_meas = simulate_clean_datacube(data['stitched'][3])
scan_mask = checkerboard_mask(baseline_meas.array.shape[:2])

tasks:   1%|          | 2/365 [00:02<06:43,  1.11s/it]

tasks:   1%|          | 2/365 [00:01<03:39,  1.65it/s]

In [ ]:
merged_opt = merge_scan(baseline_meas, excited_meas, scan_mask, dose=None)
merged_dose_h = merge_scan(baseline_meas, excited_meas, scan_mask, dose=1e6)
merged_dose_m = merge_scan(baseline_meas, excited_meas, scan_mask, dose=1e5)


: 